# SSAFY 재활용품 VQA - Colab A100/H100 상위권용 Qwen3.5-9B BF16 Notebook (Fixed)

이 노트북은 다음 전략을 한 번에 담았습니다.

- **Qwen3.5-9B** 기반 (BF16 full precision, Flash Attention 2) 객관식 VQA 파인튜닝
- **answer-only loss masking + BF16 full precision**
- **free generation 대신 logprob reranking**
- **option shuffle augmentation (epoch별 다양성)**
- **shared adapter + count expert + material expert adapter**
- **text-only prior (validation leakage 제거)**
- **dev crowd 응답 기반 pseudo label**
- **TrashNet / TACO public data synthetic QA**
- **OCR 질문용 easyocr 보조**
- **질문 유형별 앙상블 (자동 weight optimization)**

### 원본 대비 수정 사항 (7개 Fix)
1. **[P0]** answer NaN 필터 — text prior crash 방지
2. **[P1]** text prior leakage 제거 — `official_train_core`로 학습
3. **[P1]** hash() → hashlib.md5 — 세션 간 재현성 보장
4. **[P1]** Qwen3 thinking mode 비활성화 — `enable_thinking=False`
5. **[P2]** predict_dataframe branch probs 저장 + ensemble weight optimization 연결
6. **[P2]** validation split qtype 분포 체크 + 재시도
7. **[P2]** 옵션 셔플 epoch별 다양성 — Dataset에 epoch 반영

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab H100 설치 셀
# Qwen3.5 네이티브 멀티모달 모델 — 최신 transformers 소스 필요
!pip -q install -U git+https://github.com/huggingface/transformers accelerate
!pip -q install -U "peft>=0.13.2" "bitsandbytes>=0.46.1" datasets rapidfuzz easyocr

# Flash Attention 2 — 미리 빌드된 wheel로 30초 내 설치 (소스 빌드 시 20~30분 소요)
# Colab A100/H100: CUDA 12.8 + PyTorch 2.10 + Python 3.12 기준
!pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.7.16/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
import os
import gc
import re
import io
import math
import json
import glob
import time
import copy
import random
import shutil
import string
import zipfile
import warnings
import hashlib
import subprocess
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

from datasets import load_dataset

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
torch.set_float32_matmul_precision("high")

@dataclass
class CFG:
    seed: int = 42

    # 데이터 경로
    zip_path: str = ""  # 사용 안 함 (Google Drive 마운트)
    extract_dir: str = "/content/drive/MyDrive"  # Google Drive 마운트 경로
    output_dir: str = "/content/outputs_ssafy_vqa_qwen35_h100"

    # 모델
    model_id: str = "Qwen/Qwen3.5-9B"
    train_use_4bit: bool = False  # H100 80GB: BF16 full precision
    infer_use_4bit: bool = False  # BF16 추론
    train_dtype: str = "bfloat16"
    infer_dtype: str = "bfloat16"

    # Qwen3.5 동적 해상도 (H100 BF16)
    train_min_pixels: int = 512 * 512
    train_max_pixels: int = 1536 * 1536
    infer_min_pixels: int = 672 * 672
    infer_max_pixels: int = 1792 * 1792
    infer_ocr_max_pixels: int = 2048 * 2048

    # 학습
    per_device_batch_size: int = 4  # H100 BF16
    grad_accum_steps: int = 4  # effective batch=16
    num_workers: int = 2
    shared_epochs: int = 2
    count_epochs: int = 3
    lr_shared: float = 5e-5  # BF16 needs lower LR
    lr_count: float = 7e-5
    lr_material: float = 5e-5  # material expert (full data + 3x upsample)
    material_epochs: int = 2
    n_perm_material: int = 3
    use_fulltext_logprob: bool = True  # A/B: True=full-text mean-logprob, False=letter-only
    use_template_aware_cv: bool = True  # 하이브리드: template-aware group split (과낙관 방지)
    material_upsample: int = 3  # material/object 샘플 3x 복제
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0

    # LoRA
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )
    # 작게나마 도메인 적응을 위해 projector/merger 계열을 같이 푸는 옵션
    unfreeze_full_module_keywords: Tuple[str, ...] = ("merger",)

    # 데이터 증강
    shuffle_options: bool = True
    jpeg_aug: bool = True
    jpeg_aug_prob: float = 0.35
    jpeg_quality_min: int = 55
    jpeg_quality_max: int = 95

    # dev pseudo
    use_dev_pseudo: bool = True
    dev_shared_min_conf: float = 0.80   # 4/5 이상
    dev_count_min_conf: float = 0.60    # 3/5 이상
    dev_shared_weight: float = 0.35  # 하이브리드: shared에는 약하게 (ChatGPT안)
    dev_count_weight_low: float = 0.40  # 3/5 일치 → 0.40 (하이브리드)
    dev_count_weight_high: float = 0.70  # 4/5 일치 → 0.70 (하이브리드)

    # 외부 데이터
    use_trashnet: bool = True
    use_taco: bool = True
    trashnet_max_samples: int = 2500
    taco_max_samples: int = 1800
    trashnet_weight: float = 0.55
    taco_shared_weight: float = 0.45
    taco_count_weight: float = 0.75
    external_dir: str = "/content/drive/MyDrive/external_data"

    # 검증 튜닝 모드
    # --- early stopping & checkpoint ---
    early_stopping_patience: int = 2          # valid_loss 개선 없으면 N epoch 후 중단
    checkpoint_save_steps: int = 100          # N step마다 mid-epoch 체크포인트
    gdrive_backup_dir: str = "/content/drive/MyDrive/vqa_ckpt"  # Google Drive 백업 경로
    gdrive_backup_best: bool = True           # best model도 Drive에 백업

    tune_holdout: bool = True
    holdout_size: float = 0.10
    retrain_final_after_tune: bool = False

    # 추론
    n_perm_shared: int = 3
    n_perm_count: int = 5
    uncertain_margin: float = 0.12

    # OCR
    use_ocr: bool = True
    ocr_langs: Tuple[str, ...] = ("ko", "en")
    ocr_weight: float = 0.35
    upsample_ocr_text: int = 3  # OCR only 1.1%

    # 앙상블 가중치 기본값
    ensemble_weights: Dict[str, Dict[str, float]] = field(default_factory=lambda: {
        "count":    {"shared": 0.22, "count": 0.68, "text": 0.10, "ocr": 0.00, "material": 0.00},
        "material": {"shared": 0.30, "count": 0.00, "text": 0.20, "ocr": 0.05, "material": 0.45},
        "object":   {"shared": 0.30, "count": 0.00, "text": 0.20, "ocr": 0.05, "material": 0.45},
        "color":    {"shared": 0.45, "count": 0.00, "text": 0.20, "ocr": 0.10, "material": 0.25},
        "location": {"shared": 0.35, "count": 0.00, "text": 0.25, "ocr": 0.25, "material": 0.15},
        "ocr":      {"shared": 0.55, "count": 0.00, "text": 0.10, "ocr": 0.35, "material": 0.00},
        "default":  {"shared": 0.50, "count": 0.00, "text": 0.20, "ocr": 0.00, "material": 0.30},
    })

CFG = CFG()

# Attention: Flash Attention 2 (pre-built wheel 설치, ~20% 학습 속도 향상)
CFG.attn_implementation = "flash_attention_2"

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.external_dir).mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda:", torch.version.cuda)
    print("bf16 supported:", torch.cuda.is_bf16_supported())

if "google.colab" in str(get_ipython()):
    from google.colab import drive

In [ ]:
# ============ 데이터 준비 (Google Drive 마운트) ============
# 데이터는 이미 Google Drive에 있음
# /content/drive/MyDrive/ 아래에 train.csv, test.csv, dev.csv + train/, test/, dev/ 폴더

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

data_root = Path(CFG.extract_dir)

# CSV 로드
train_csv_path = data_root / "train.csv"
test_csv_path = data_root / "test.csv"
dev_csv_path = data_root / "dev.csv"
sample_submission_path = data_root / "sample_submission.csv"

for p in [train_csv_path, test_csv_path, dev_csv_path]:
    assert p.exists(), f"파일 없음: {p}"

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
dev_df = pd.read_csv(dev_csv_path)

# [Fix 1] answer 결측 필터 — NaN/빈값으로 text prior crash 방지
train_df = train_df[train_df["answer"].isin(["a", "b", "c", "d"])].reset_index(drop=True)
print(f"train after answer filter: {len(train_df)}")

# 이미지 경로를 절대 경로로 변환
for df in [train_df, test_df, dev_df]:
    df["path"] = df["path"].apply(lambda x: str(data_root / x))

print(f"train: {len(train_df)}, test: {len(test_df)}, dev: {len(dev_df)}")
print(f"data_root: {data_root}")
print(f"첫 번째 이미지 존재: {Path(train_df['path'].iloc[0]).exists()}")

# 외부 데이터 디렉토리
Path(CFG.external_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# ============ dev pseudo + 외부 데이터 ============

LETTERS = ["a", "b", "c", "d"]

def normalize_text(x: Any) -> str:
    x = str(x).lower()
    x = re.sub(r"[^0-9a-z가-힣]+", "", x)
    return x


def resolve_image_path(rel_path: str) -> str:
    p = str(rel_path)
    if os.path.isabs(p):
        return p
    return str(Path(CFG.extract_dir) / p)

def normalize_answer_letter(x) -> Optional[str]:
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    if x in LETTERS:
        return x
    try:
        idx = int(float(x)) - 1
        if 0 <= idx < 4:
            return LETTERS[idx]
    except (ValueError, TypeError):
        pass
    return None

def detect_qtype(question: str) -> str:
    q = str(question).strip()
    if re.search(r'몇\s*[개병캔컵장박스]|개수|총\s*몇|얼마나\s*많', q):
        return "count"
    if re.search(r'글자|텍스트|문자|라벨|상표|브랜드|제품명|적[혀힌]|쓰[여인]|인쇄|읽|로고', q):
        return "ocr"
    if re.search(r'색[상깔]?[은이가]|컬러|무슨\s*색', q):
        return "color"
    if re.search(r'어디|위치|장소|놓[여인]|위에|아래|옆에|안에|밖에', q):
        return "location"
    if re.search(r'재질|소재|재료|만들어[진]|어떤\s*재', q):
        return "material"
    if re.search(r'무엇|어떤\s*[물종]|뭐[가인]', q):
        return "object"
    return "material"

def detect_qtype_detail(question: str) -> str:
    """24개 세부 질문 유형 분류 (Discussion 그래프 기반).
    Top 5: material_general(21.9%), object_type(18.8%), count_bottle(9.8%), count_general(8.1%), count_cup(6.6%)
    """
    q = str(question).strip()

    # count 세부
    if re.search(r'몇\s*[개병캔컵장박스]|개수|총\s*몇|얼마나\s*많', q):
        if re.search(r'병|페트|PET', q): return "count_bottle"
        if re.search(r'컵|잔|텀블러', q): return "count_cup"
        if re.search(r'박스|상자|종이박스', q): return "count_box"
        if re.search(r'캔', q): return "count_can"
        if re.search(r'뚜껑|마개|캡', q): return "count_lid"
        return "count_general"

    # ocr 세부
    if re.search(r'글자|텍스트|문자|라벨|상표|브랜드|제품명|적[혀힌]|쓰[여인]|인쇄|읽|로고', q):
        if re.search(r'브랜드|상표|제품명|회사', q): return "brand_or_product"
        if re.search(r'맛|향|flavor', q): return "flavor"
        return "text_or_symbol"

    # color 세부
    if re.search(r'색|컬러|color|무슨\s*빛', q):
        if re.search(r'뚜껑|마개|캡', q): return "color_lid"
        if re.search(r'빨대|스트로우', q): return "color_straw"
        if re.search(r'라벨', q): return "color_label"
        return "color_general"

    # location
    if re.search(r'어디|위치|장소|어느\s*곳|놓[여인]|위[에치]|아래|옆', q):
        return "position"

    # object vs material
    if re.search(r'종류|무엇|어떤\s*물[체건]|물체|뭐', q):
        if re.search(r'대부분|가장\s*많|주로', q): return "majority_type"
        return "object_type"

    if re.search(r'분리수거|재활용\s*분류|어떤\s*[쓰통]', q):
        return "recycling_class"

    # material 세부
    if re.search(r'재[질료]|소재|만들어[진진]|재활용', q):
        if re.search(r'포장|패키지', q): return "material_package"
        if re.search(r'컵|잔', q): return "material_cup"
        if re.search(r'병', q): return "material_bottle"
        if re.search(r'캔', q): return "material_can"
        return "material_general"

    return "material_general"


def build_dev_pseudo(dev_csv_path: str) -> pd.DataFrame:
    raw = pd.read_csv(dev_csv_path)
    raw["path"] = raw["path"].map(resolve_image_path)
    raw["question"] = raw["question"].astype(str).str.strip()
    for col in ["a", "b", "c", "d"]:
        raw[col] = raw[col].astype(str).str.strip()

    answers = ["answer1", "answer2", "answer3", "answer4", "answer5"]
    maj_answers = []
    maj_conf = []
    for _, row in raw.iterrows():
        votes = []
        for c in answers:
            v = normalize_answer_letter(row.get(c))
            if v is not None:
                votes.append(v)
        if not votes:
            maj_answers.append(None)
            maj_conf.append(0.0)
            continue
        vc = pd.Series(votes).value_counts()
        maj_answers.append(vc.index[0])
        maj_conf.append(vc.iloc[0] / len(votes))

    df = raw[["id", "path", "question", "a", "b", "c", "d"]].copy()
    df["answer"] = maj_answers
    df["vote_conf"] = maj_conf
    df["qtype"] = df["question"].map(detect_qtype)
    df["source"] = "dev_pseudo"
    df["sample_weight"] = 0.0
    return df[df["answer"].isin(LETTERS)].reset_index(drop=True)

dev_pseudo_df = build_dev_pseudo(dev_csv_path)
print("dev_pseudo:", dev_pseudo_df.shape)
display(dev_pseudo_df.head(3))

# -------- TrashNet -> material synthetic --------
MATERIAL_CHOICES = ["유리", "금속", "종이", "플라스틱", "비닐"]
TRASHNET_LABEL_TO_MATERIAL = {
    "glass": "유리",
    "metal": "금속",
    "paper": "종이",
    "cardboard": "종이",
    "plastic": "플라스틱",
}
MATERIAL_Q_TEMPLATES = [
    "사진 속 재활용품의 주된 재질은 무엇인가요?",
    "사진에 보이는 재활용 가능한 물체의 소재는 무엇인가요?",
    "이미지 속 물체는 어떤 재질의 재활용품인가요?",
    "사진에 보이는 물체의 재질은 무엇인가요?",
]

def make_mc_options(correct: str, pool: List[str], rng: random.Random) -> Tuple[List[str], str]:
    negatives = [x for x in pool if x != correct]
    opts = rng.sample(negatives, 3) + [correct]
    rng.shuffle(opts)
    answer = LETTERS[opts.index(correct)]
    return opts, answer

def maybe_download_trashnet_zip() -> str:
    out_dir = Path(CFG.external_dir) / "trashnet"
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir / "dataset-resized.zip"
    extracted_dir = out_dir / "dataset-resized"

    if extracted_dir.exists() and len(list(extracted_dir.rglob("*.jpg"))) > 100:
        return str(extracted_dir)

    if not zip_path.exists():
        url = "https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip"
        print("downloading TrashNet resized zip...")
        subprocess.run(["wget", "-q", "-O", str(zip_path), url], check=True)

    print("extracting TrashNet...")
    shutil.unpack_archive(str(zip_path), str(out_dir))
    return str(extracted_dir)

def build_trashnet_synthetic(max_samples: int = 2500, seed: int = 42) -> pd.DataFrame:
    extracted_dir = maybe_download_trashnet_zip()
    rng = random.Random(seed)

    rows = []
    image_paths = []
    for class_dir in sorted(Path(extracted_dir).iterdir()):
        if class_dir.is_dir():
            for p in class_dir.rglob("*.jpg"):
                image_paths.append((class_dir.name.lower(), str(p)))

    rng.shuffle(image_paths)
    kept = 0
    for label_name, img_path in tqdm(image_paths, desc="TrashNet synthetic"):
        if label_name not in TRASHNET_LABEL_TO_MATERIAL:
            continue

        correct = TRASHNET_LABEL_TO_MATERIAL[label_name]
        opts, ans = make_mc_options(correct, MATERIAL_CHOICES, rng)
        q = rng.choice(MATERIAL_Q_TEMPLATES)

        rows.append({
            "id": f"trashnet_{kept:05d}",
            "path": img_path,
            "question": q,
            "a": opts[0],
            "b": opts[1],
            "c": opts[2],
            "d": opts[3],
            "answer": ans,
            "qtype": "material",
            "source": "trashnet",
            "sample_weight": CFG.trashnet_weight,
        })
        kept += 1
        if kept >= max_samples:
            break

    return pd.DataFrame(rows)

# -------- TACO -> count/material synthetic --------
TACO_COUNT_TEMPLATES = [
    "사진에 보이는 재활용 가능한 {target}은 몇 개입니까?",
    "사진 속 {target}은 총 몇 개인가요?",
    "이미지에서 확인되는 재활용 가능한 {target}의 개수는 몇 개인가요?",
]
TACO_MATERIAL_TEMPLATES = [
    "사진 속 가장 눈에 띄는 재활용품의 주된 재질은 무엇인가요?",
    "이미지에 보이는 주요 재활용품의 재질은 무엇인가요?",
]

def maybe_clone_taco() -> str:
    repo_dir = Path(CFG.external_dir) / "TACO"
    if repo_dir.exists():
        return str(repo_dir)
    print("cloning TACO repo...")
    subprocess.run(["git", "clone", "https://github.com/pedropro/TACO.git", str(repo_dir)], check=True)
    return str(repo_dir)

def maybe_download_taco_images(repo_dir: str):
    data_dir = Path(repo_dir) / "data"
    jpgs = list(data_dir.rglob("*.jpg"))
    if len(jpgs) > 100:
        print("TACO images already downloaded:", len(jpgs))
        return
    print("downloading TACO images...")
    subprocess.run(["python", "download.py"], cwd=repo_dir, check=True)

def find_taco_annotation_json(repo_dir: str) -> str:
    candidates = []
    for p in Path(repo_dir).rglob("*.json"):
        name = p.name.lower()
        if "annotation" in name and "unofficial" not in name:
            candidates.append(str(p))
    if not candidates:
        raise FileNotFoundError("TACO annotation json not found")
    candidates = sorted(candidates, key=len)
    return candidates[0]

def map_taco_category(cat_name: str, supercategory: str) -> Optional[Dict[str, str]]:
    s = f"{supercategory} {cat_name}".lower()
    s = s.replace("-", " ").replace("_", " ")
    if "drink can" in s or re.search(r"\bcan\b", s):
        return {"group_ko": "금속 캔", "material_ko": "금속"}
    if "glass" in s and "bottle" in s:
        return {"group_ko": "유리 병", "material_ko": "유리"}
    if "bottle" in s:
        return {"group_ko": "플라스틱 병", "material_ko": "플라스틱"}
    if "paper cup" in s:
        return {"group_ko": "종이컵", "material_ko": "종이"}
    if "cup" in s:
        return {"group_ko": "플라스틱 컵", "material_ko": "플라스틱"}
    if "carton" in s or "tetrapak" in s or "tetra pak" in s:
        return {"group_ko": "종이 팩", "material_ko": "종이"}
    if "bag" in s or "wrapper" in s or "film" in s:
        return {"group_ko": "비닐", "material_ko": "비닐"}
    if "paper" in s or "cardboard" in s:
        return {"group_ko": "종이", "material_ko": "종이"}
    return None

def make_count_options(correct_count: int, rng: random.Random) -> Tuple[List[str], str]:
    cand = [correct_count]
    for d in rng.sample([-2, -1, 1, 2, 3], 5):
        v = correct_count + d
        if v >= 1 and v not in cand:
            cand.append(v)
        if len(cand) >= 4:
            break
    while len(cand) < 4:
        v = max(1, correct_count + rng.randint(-3, 3))
        if v not in cand:
            cand.append(v)
    vals = cand[:4]
    rng.shuffle(vals)
    opts = [f"{v}개" for v in vals]
    ans = LETTERS[vals.index(correct_count)]
    return opts, ans

def build_taco_synthetic(max_samples: int = 1800, seed: int = 42) -> pd.DataFrame:
    repo_dir = maybe_clone_taco()
    maybe_download_taco_images(repo_dir)
    ann_path = find_taco_annotation_json(repo_dir)
    print("TACO annotation:", ann_path)

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    categories = {c["id"]: c for c in coco["categories"]}
    images = {img["id"]: img for img in coco["images"]}

    image_name_to_path = {p.name: str(p) for p in Path(repo_dir).rglob("*.jpg")}

    mapped_anns = []
    for ann in coco["annotations"]:
        cat = categories[ann["category_id"]]
        mapped = map_taco_category(cat.get("name", ""), cat.get("supercategory", ""))
        if mapped is None:
            continue
        mapped_anns.append({
            "image_id": ann["image_id"],
            "group_ko": mapped["group_ko"],
            "material_ko": mapped["material_ko"],
        })
    mapped_df = pd.DataFrame(mapped_anns)
    if mapped_df.empty:
        return pd.DataFrame(columns=["id","path","question","a","b","c","d","answer","qtype","source","sample_weight"])

    rng = random.Random(seed)
    rows = []

    # count synthetic
    group_counts = (
        mapped_df.groupby(["image_id", "group_ko"]).size().reset_index(name="count")
    )
    group_counts = group_counts[group_counts["count"].between(1, 6)].sample(frac=1.0, random_state=seed)

    for _, row in tqdm(group_counts.iterrows(), total=len(group_counts), desc="TACO count synthetic"):
        img_meta = images.get(int(row["image_id"]))
        if img_meta is None:
            continue

        file_name = img_meta.get("file_name", "")
        img_path = image_name_to_path.get(file_name)
        if img_path is None:
            continue

        opts, ans = make_count_options(int(row["count"]), rng)
        q = rng.choice(TACO_COUNT_TEMPLATES).format(target=row["group_ko"])
        rows.append({
            "id": f"taco_count_{int(row['image_id'])}_{normalize_text(row['group_ko'])}",
            "path": img_path,
            "question": q,
            "a": opts[0],
            "b": opts[1],
            "c": opts[2],
            "d": opts[3],
            "answer": ans,
            "qtype": "count",
            "source": "taco",
            "sample_weight": CFG.taco_count_weight,
        })
        if len(rows) >= max_samples:
            break

    # material synthetic 조금 추가
    dominant = (
        mapped_df.groupby(["image_id", "material_ko"]).size().reset_index(name="n")
        .sort_values(["image_id", "n"], ascending=[True, False])
        .drop_duplicates("image_id")
    )
    material_rows = 0
    for _, row in dominant.sample(frac=1.0, random_state=seed).iterrows():
        if material_rows >= max_samples // 4:
            break

        img_meta = images.get(int(row["image_id"]))
        if img_meta is None:
            continue
        file_name = img_meta.get("file_name", "")
        img_path = image_name_to_path.get(file_name)
        if img_path is None:
            continue

        correct = row["material_ko"]
        opts, ans = make_mc_options(correct, MATERIAL_CHOICES, rng)
        q = rng.choice(TACO_MATERIAL_TEMPLATES)
        rows.append({
            "id": f"taco_mat_{int(row['image_id'])}",
            "path": img_path,
            "question": q,
            "a": opts[0],
            "b": opts[1],
            "c": opts[2],
            "d": opts[3],
            "answer": ans,
            "qtype": "material",
            "source": "taco",
            "sample_weight": CFG.taco_shared_weight,
        })
        material_rows += 1

    return pd.DataFrame(rows)

trashnet_df = pd.DataFrame()
taco_df = pd.DataFrame()

if CFG.use_trashnet:
    try:
        trashnet_df = build_trashnet_synthetic(CFG.trashnet_max_samples, CFG.seed)
        print("trashnet_df:", trashnet_df.shape)
    except Exception as e:
        print("TrashNet synthetic failed:", repr(e))

if CFG.use_taco:
    try:
        taco_df = build_taco_synthetic(CFG.taco_max_samples, CFG.seed)
        print("taco_df:", taco_df.shape)
    except Exception as e:
        print("TACO synthetic failed:", repr(e))

display(trashnet_df.head(2) if len(trashnet_df) else pd.DataFrame())
display(taco_df.head(2) if len(taco_df) else pd.DataFrame())

In [ ]:
# ============ 학습 풀 구성 + text prior ============

# qtype 컬럼 추가 (detect_qtype은 Cell 6에서 정의됨)
for _df in [train_df, test_df, dev_df]:
    if "qtype" not in _df.columns:
        _df["qtype"] = _df["question"].map(detect_qtype)
print("train_df qtype dist:\n", train_df["qtype"].value_counts())
def stratify_key(df: pd.DataFrame) -> pd.Series:
    key = df["qtype"].astype(str) + "_" + df["answer"].astype(str)
    vc = key.value_counts()
    # 너무 희소한 strata는 qtype만 사용
    key = np.where(key.map(vc) >= 2, key, df["qtype"].astype(str))
    return pd.Series(key, index=df.index)

def normalize_question_template(q: str) -> str:
    """질문 문자열을 정규화하여 템플릿 그룹으로 매핑.
    숫자, 구체적 물체명, 색상 등을 와일드카드로 치환."""
    q = str(q).strip()
    q = re.sub(r'\d+', 'N', q)  # 숫자 → N
    q = re.sub(r'플라스틱\s*병|유리\s*병|페트\s*병|캔|종이\s*컵|종이\s*팩|텀블러', 'OBJ', q)
    q = re.sub(r'빨간|파란|노란|초록|검[정은]|하[얀얀]|투명|갈색', 'COLOR', q)
    q = re.sub(r'유리|금속|플라스틱|종이|비닐|스티로폼|알루미늄', 'MAT', q)
    return q.strip()

def template_aware_group_split(df, test_size=0.15, seed=42, max_retries=10):
    """[Fix 6] Template-aware group split + qtype 분포 체크.
    qtype 분포 왜곡이 5% 이내가 될 때까지 seed를 바꿔가며 재시도."""
    from sklearn.model_selection import GroupShuffleSplit

    df = df.copy()
    df["_template"] = df["question"].map(normalize_question_template)
    overall_dist = df["qtype"].value_counts(normalize=True)

    best_train, best_val = None, None
    best_skew = float("inf")
    best_attempt = 0

    for attempt in range(max_retries):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed + attempt)
        train_idx, val_idx = next(gss.split(df, groups=df["_template"]))

        val_dist = df.iloc[val_idx]["qtype"].value_counts(normalize=True)
        skew = max(abs(val_dist.get(qt, 0) - overall_dist.get(qt, 0)) for qt in overall_dist.index)

        if skew < best_skew:
            best_skew = skew
            best_train = train_idx
            best_val = val_idx
            best_attempt = attempt

        if skew < 0.05:
            break

    train_df = df.iloc[best_train].drop(columns=["_template"]).reset_index(drop=True)
    val_df = df.iloc[best_val].drop(columns=["_template"]).reset_index(drop=True)

    print(f"template-aware split (attempt {best_attempt+1}/{max_retries}, max_skew={best_skew:.3f}): train={len(train_df)}, val={len(val_df)}")
    print(f"  val qtype dist: {df.iloc[best_val]['qtype'].value_counts(normalize=True).round(3).to_dict()}")
    print(f"  overall dist:   {overall_dist.round(3).to_dict()}")
    return train_df, val_df

valid_df = None
official_train_core = train_df.copy().reset_index(drop=True)

if CFG.tune_holdout:
    if getattr(CFG, "use_template_aware_cv", True):
        official_train_core, valid_df = template_aware_group_split(
            train_df, test_size=CFG.holdout_size, seed=CFG.seed
        )
    else:
        official_train_core, valid_df = train_test_split(
            train_df,
            test_size=CFG.holdout_size,
            random_state=CFG.seed,
            stratify=stratify_key(train_df),
        )
    official_train_core = official_train_core.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    print("train_core:", official_train_core.shape, "valid:", valid_df.shape)
else:
    print("final mode: using full official train")

shared_parts = [official_train_core.assign(sample_weight=1.0, source="official")]
count_parts = [official_train_core[official_train_core["qtype"] == "count"].assign(sample_weight=1.0, source="official_count")]

if CFG.use_dev_pseudo and not dev_pseudo_df.empty:
    dev_shared = dev_pseudo_df[dev_pseudo_df["vote_conf"] >= CFG.dev_shared_min_conf].copy()
    dev_shared["sample_weight"] = CFG.dev_shared_weight
    if not dev_shared.empty:
        shared_parts.append(dev_shared)

    dev_count = dev_pseudo_df[
        (dev_pseudo_df["qtype"] == "count") &
        (dev_pseudo_df["vote_conf"] >= CFG.dev_count_min_conf)
    ].copy()
    if not dev_count.empty:
        dev_count["sample_weight"] = np.where(
            dev_count["vote_conf"] >= 0.80,
            CFG.dev_count_weight_high,
            CFG.dev_count_weight_low,
        )
        count_parts.append(dev_count)

if len(trashnet_df):
    shared_parts.append(trashnet_df.copy())

if len(taco_df):
    shared_parts.append(taco_df.copy())
    count_parts.append(taco_df[taco_df["qtype"] == "count"].copy())

shared_train_df = pd.concat(shared_parts, ignore_index=True)
count_train_df = pd.concat(count_parts, ignore_index=True)

# ---- material/object expert 학습 데이터 (full data + material/object 3x upsampling) ----
# dev 데이터는 count-heavy(71.4%)라 material expert에서 dev weight를 더 낮춤
_shared_for_mat = shared_train_df.copy()
_dev_mask = _shared_for_mat["source"].str.startswith("dev_pseudo") if "source" in _shared_for_mat.columns else pd.Series(False, index=_shared_for_mat.index)
_shared_for_mat.loc[_dev_mask, "sample_weight"] = _shared_for_mat.loc[_dev_mask, "sample_weight"] * 0.5  # dev 영향 반감
material_parts = [_shared_for_mat]  # dev 약화된 전체 데이터를 기반으로
_mat_obj = shared_train_df[shared_train_df["qtype"].isin(["material", "object"])].copy()
if len(_mat_obj) > 0 and CFG.material_upsample > 1:
    for _ in range(CFG.material_upsample - 1):  # 3x = 원본 + 2배 추가
        material_parts.append(_mat_obj)
    print(f"material/object upsampled: {len(_mat_obj)} x{CFG.material_upsample}")
material_train_df = pd.concat(material_parts, ignore_index=True)
material_train_df = material_train_df[material_train_df["answer"].isin(LETTERS)].reset_index(drop=True)

print("material_train_df:", material_train_df.shape)
if "source" in material_train_df.columns:
    print(material_train_df["source"].value_counts())
print("material_train_df qtype dist:")
print(material_train_df["qtype"].value_counts())

# OCR upsample (only 1.1% → address underrepresentation)
if hasattr(CFG, "upsample_ocr_text") and CFG.upsample_ocr_text > 1:
    _ocr = shared_train_df[shared_train_df["qtype"] == "ocr"]
    if len(_ocr) > 0:
        shared_train_df = pd.concat([shared_train_df] + [_ocr] * (CFG.upsample_ocr_text - 1), ignore_index=True)
        print(f"OCR upsampled: {len(_ocr)} x{CFG.upsample_ocr_text}")

shared_train_df = shared_train_df[shared_train_df["answer"].isin(LETTERS)].reset_index(drop=True)
count_train_df = count_train_df[count_train_df["answer"].isin(LETTERS)].reset_index(drop=True)

print("shared_train_df:", shared_train_df.shape)
print(shared_train_df["source"].value_counts())
print("count_train_df :", count_train_df.shape)
print(count_train_df["source"].value_counts())

# ---------- text prior ----------
def compose_text_features(df: pd.DataFrame) -> List[str]:
    out = []
    for _, row in df.iterrows():
        out.append(
            f"[Q] {row['question']} "
            f"[A] {row['a']} [B] {row['b']} [C] {row['c']} [D] {row['d']} "
            f"[QTYPE] {row['qtype']}"
        )
    return out

text_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 6),
    min_df=2,
    sublinear_tf=True,
)

text_clf = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    max_iter=4000,
    random_state=CFG.seed,
)

# [Fix 2] text prior leakage 제거 — official_train_core만 사용 (valid_df 미포함)
X_text_train = text_vectorizer.fit_transform(compose_text_features(official_train_core))
y_text_train = official_train_core["answer"].values
text_clf.fit(X_text_train, y_text_train)

text_classes_ = list(text_clf.classes_)
print("text classes:", text_classes_)

def predict_text_proba(df: pd.DataFrame) -> np.ndarray:
    X = text_vectorizer.transform(compose_text_features(df))
    probs = text_clf.predict_proba(X)
    order = [text_classes_.index(c) for c in LETTERS]
    return probs[:, order]

train_text_probs = predict_text_proba(train_df.head(5))
print(train_text_probs.shape)

In [ ]:
# ============ 모델/학습 헬퍼 ============
SYSTEM_PROMPT_GENERAL = (
    "너는 재활용품 이미지 기반 객관식 VQA 전문가다. "
    "이미지와 질문, 선택지를 보고 정답 하나를 고른다. "
    "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력해야 한다."
)

SYSTEM_PROMPT_COUNT = (
    "너는 재활용품 개수 세기 전문가다. "
    "질문에서 지정한 대상만 정확히 세고, 비슷하지만 다른 물체는 제외한다. "
    "부분적으로 보이더라도 대상이 명확하면 포함한다. "
    "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력해야 한다."
)

SYSTEM_PROMPT_MATERIAL = (
    "너는 재활용 소재 및 물체 분류 전문가다. "
    "유리, 플라스틱, 금속, 종이, 비닐 등 재질을 정확히 판별하고, "
    "캔, 유리병, 페트병, 종이팩 등 물체 종류도 구분한다. "
    "재질의 시각적 특성(투명도, 광택, 질감, 색상)에 주의한다. "
    "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력해야 한다."
)


# [Fix 4 safety] enable_thinking=False를 안전하게 적용하는 래퍼
# Qwen3 chat template이 enable_thinking을 지원하지 않으면 kwarg 없이 재시도
_ENABLE_THINKING_SUPPORTED = None  # lazy detection

def safe_apply_chat_template(processor, messages, tokenize=False, add_generation_prompt=True):
    global _ENABLE_THINKING_SUPPORTED
    if _ENABLE_THINKING_SUPPORTED is None:
        try:
            result = processor.apply_chat_template(
                messages, tokenize=tokenize, add_generation_prompt=add_generation_prompt,
                enable_thinking=False,
            )
            _ENABLE_THINKING_SUPPORTED = True
            return result
        except (TypeError, Exception):
            _ENABLE_THINKING_SUPPORTED = False
            print("[Fix 4] enable_thinking not supported by this model/template — disabled")
            return processor.apply_chat_template(
                messages, tokenize=tokenize, add_generation_prompt=add_generation_prompt,
            )
    elif _ENABLE_THINKING_SUPPORTED:
        return processor.apply_chat_template(
            messages, tokenize=tokenize, add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    else:
        return processor.apply_chat_template(
            messages, tokenize=tokenize, add_generation_prompt=add_generation_prompt,
        )

def build_user_prompt(question: str, options: List[str]) -> str:
    return (
        f"질문: {question}\n"
        f"선택지:\n"
        f"a. {options[0]}\n"
        f"b. {options[1]}\n"
        f"c. {options[2]}\n"
        f"d. {options[3]}\n\n"
        f"정답 문자 하나만 답하세요."
    )

def random_jpeg_aug(img: Image.Image, rng: random.Random) -> Image.Image:
    if (not CFG.jpeg_aug) or (rng.random() > CFG.jpeg_aug_prob):
        return img
    q = rng.randint(CFG.jpeg_quality_min, CFG.jpeg_quality_max)
    bio = io.BytesIO()
    img.save(bio, format="JPEG", quality=q)
    bio.seek(0)
    return Image.open(bio).convert("RGB")

def maybe_shuffle_options(row: pd.Series, rng: random.Random, train: bool) -> Tuple[List[str], str, List[int]]:
    options = [str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])]
    correct_idx = LETTERS.index(str(row["answer"]))
    perm = [0, 1, 2, 3]
    if train and CFG.shuffle_options:
        rng.shuffle(perm)
    shuffled = [options[i] for i in perm]
    new_answer = LETTERS[perm.index(correct_idx)]
    return shuffled, new_answer, perm

class MCVQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool, system_prompt: str, seed: int = 42):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.system_prompt = system_prompt
        self.seed = seed
        self.epoch = 0  # [Fix 7] epoch별 다양한 옵션 셔플

    def set_epoch(self, epoch: int):
        """[Fix 7] epoch마다 다른 셔플을 생성하도록 epoch 설정."""
        self.epoch = epoch

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        rng = random.Random(self.seed + idx + self.epoch * len(self.df) + (13 if self.train else 0))  # [Fix 7]
        img = Image.open(row["path"]).convert("RGB")
        if self.train:
            img = random_jpeg_aug(img, rng)

        options, answer_letter, _ = maybe_shuffle_options(row, rng, self.train)
        user_text = build_user_prompt(row["question"], options)

        prefix_messages = [
            {"role": "system", "content": [{"type": "text", "text": self.system_prompt}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ]},
        ]
        full_messages = prefix_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": answer_letter}]}
        ]

        return {
            "id": row["id"],
            "image": img,
            "prefix_messages": prefix_messages,
            "full_messages": full_messages,
            "answer": answer_letter,
            "sample_weight": float(row.get("sample_weight", 1.0)),
        }

class DataCollatorForQwenVL:
    def __init__(self, processor: Any):
        self.processor = processor

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [b["image"] for b in batch]
        prefix_texts = [
            safe_apply_chat_template(self.processor, b["prefix_messages"], tokenize=False, add_generation_prompt=True)
            for b in batch
        ]
        full_texts = [
            safe_apply_chat_template(self.processor, b["full_messages"], tokenize=False, add_generation_prompt=False)
            for b in batch
        ]

        enc_full = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        enc_prefix = self.processor(
            text=prefix_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = enc_full["input_ids"].clone()
        prefix_lens = enc_prefix["attention_mask"].sum(dim=1).tolist()
        for i, p_len in enumerate(prefix_lens):
            labels[i, :int(p_len)] = -100
            labels[i, enc_full["attention_mask"][i] == 0] = -100

        enc_full["labels"] = labels
        enc_full["sample_weight"] = torch.tensor(
            [b["sample_weight"] for b in batch], dtype=torch.float32
        )
        return enc_full

def get_dtype(dtype_name: str) -> torch.dtype:
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    return torch.float32

def build_processor(train: bool = True, ocr_mode: bool = False):
    if train:
        min_pixels = CFG.train_min_pixels
        max_pixels = CFG.train_max_pixels
    else:
        min_pixels = CFG.infer_min_pixels
        max_pixels = CFG.infer_ocr_max_pixels if ocr_mode else CFG.infer_max_pixels

    processor = AutoProcessor.from_pretrained(
        CFG.model_id,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
        trust_remote_code=True,
    )
    try:
        processor.tokenizer.padding_side = "right"
    except Exception:
        pass
    return processor

def build_quant_config(use_4bit: bool, dtype_name: str):
    if not use_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=get_dtype(dtype_name),
    )

def load_base_model(for_train: bool, use_4bit: bool, dtype_name: str):
    quant_config = build_quant_config(use_4bit, dtype_name)
    kwargs = dict(
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        torch_dtype=get_dtype(dtype_name),
    )
    if quant_config is not None:
        kwargs["quantization_config"] = quant_config
        kwargs["device_map"] = {"": 0}
    else:
        kwargs["device_map"] = {"": 0}

    model = AutoModelForImageTextToText.from_pretrained(
        CFG.model_id,
        attn_implementation=getattr(CFG, "attn_implementation", "sdpa"),
        **kwargs,
    )
    if for_train:
        if use_4bit:
            model = prepare_model_for_kbit_training(model)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.config.use_cache = False
    else:
        model.config.use_cache = True
    return model

def apply_lora(model):
    lora_config = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        bias="none",
        target_modules=list(CFG.lora_target_modules),
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # projector/merger 계열은 도메인 적응을 위해 full trainable로 열기
    if CFG.unfreeze_full_module_keywords:
        for name, param in model.named_parameters():
            if any(k in name for k in CFG.unfreeze_full_module_keywords):
                param.requires_grad = True

    return model

def move_batch_to_device(batch: Dict[str, Any], device: str) -> Dict[str, Any]:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out

def compute_weighted_loss(logits: torch.Tensor, labels: torch.Tensor, sample_weight: torch.Tensor) -> torch.Tensor:
    # causal LM shift
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    vocab_size = shift_logits.size(-1)

    token_loss = F.cross_entropy(
        shift_logits.view(-1, vocab_size),
        shift_labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(shift_labels.size())

    token_mask = (shift_labels != -100).float()
    per_sample = (token_loss * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp_min(1.0)
    weighted = (per_sample * sample_weight).sum() / sample_weight.sum().clamp_min(1e-6)
    return weighted

def evaluate_loss(model, loader, device: str) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="valid", leave=False):
            batch = move_batch_to_device(batch, device)
            labels = batch.pop("labels")
            sample_weight = batch.pop("sample_weight")
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = compute_weighted_loss(outputs.logits, labels, sample_weight)
            losses.append(float(loss.item()))
    model.train()
    return float(np.mean(losses)) if losses else np.nan

def train_one_adapter(
    train_pool_df: pd.DataFrame,
    adapter_name: str,
    system_prompt: str,
    epochs: int,
    lr: float,
    valid_df: Optional[pd.DataFrame] = None,
) -> str:
    print(f"\\n===== train adapter: {adapter_name} =====")
    processor = build_processor(train=True)
    model = load_base_model(for_train=True, use_4bit=CFG.train_use_4bit, dtype_name=CFG.train_dtype)
    model = apply_lora(model)
    model.print_trainable_parameters()

    train_ds = MCVQADataset(train_pool_df, train=True, system_prompt=system_prompt, seed=CFG.seed)
    train_loader = DataLoader(
        train_ds,
        batch_size=CFG.per_device_batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=True,
        collate_fn=DataCollatorForQwenVL(processor),
        persistent_workers=CFG.num_workers > 0,
    )

    valid_loader = None
    if valid_df is not None and len(valid_df):
        valid_ds = MCVQADataset(valid_df, train=False, system_prompt=system_prompt, seed=CFG.seed)
        valid_loader = DataLoader(
            valid_ds,
            batch_size=CFG.per_device_batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=True,
            collate_fn=DataCollatorForQwenVL(processor),
            persistent_workers=CFG.num_workers > 0,
        )

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=CFG.weight_decay,
    )

    total_steps = epochs * math.ceil(len(train_loader) / CFG.grad_accum_steps)
    warmup_steps = max(1, int(total_steps * CFG.warmup_ratio))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    save_dir = Path(CFG.output_dir) / adapter_name
    save_dir.mkdir(parents=True, exist_ok=True)

    best_metric = float("inf")
    patience_counter = 0
    global_step = 0
    gdrive_base = Path(CFG.gdrive_backup_dir)

    def _backup_to_gdrive(src_path: Path, dest_name: str):
        """Google Drive에 체크포인트 백업 (실패해도 학습 중단 안 함)"""
        try:
            _gd = gdrive_base / adapter_name / dest_name
            if gdrive_base.parent.exists():
                import shutil as _sh
                _gd.mkdir(parents=True, exist_ok=True)
                _sh.copytree(src_path, _gd, dirs_exist_ok=True)
                print(f"  [gdrive backup] {_gd}")
        except Exception as e:
            print(f"  [gdrive backup failed] {repr(e)}")

    model.train()
    stopped_early = False
    for epoch in range(epochs):
        if hasattr(train_ds, "set_epoch"):
            train_ds.set_epoch(epoch)  # [Fix 7]
        running = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"{adapter_name} epoch {epoch+1}/{epochs}")
        for step, batch in enumerate(pbar, start=1):
            batch = move_batch_to_device(batch, device)
            labels = batch.pop("labels")
            sample_weight = batch.pop("sample_weight")

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = compute_weighted_loss(outputs.logits, labels, sample_weight)
                loss = loss / CFG.grad_accum_steps

            loss.backward()
            running += float(loss.item())

            if step % CFG.grad_accum_steps == 0 or step == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                pbar.set_postfix(loss=f"{running:.4f}", step=global_step)
                running = 0.0

                # Mid-epoch checkpoint (Colab disconnect protection)
                if global_step % CFG.checkpoint_save_steps == 0:
                    _ckpt = save_dir / f"step_{global_step}"
                    _ckpt.mkdir(parents=True, exist_ok=True)
                    model.save_pretrained(_ckpt)
                    _backup_to_gdrive(_ckpt, f"step_{global_step}")

        # --- epoch-end validation & early stopping ---
        valid_loss = np.nan
        if valid_loader is not None:
            valid_loss = evaluate_loss(model, valid_loader, device)
            print(f"[{adapter_name}] epoch={epoch+1} valid_loss={valid_loss:.5f}")

        metric = valid_loss if not np.isnan(valid_loss) else (epoch + 1) * 0.0
        if valid_loader is None or metric < best_metric:
            if not np.isnan(metric):
                best_metric = metric
            patience_counter = 0
            model.save_pretrained(save_dir)
            processor.save_pretrained(save_dir)
            with open(save_dir / "train_args.json", "w", encoding="utf-8") as f:
                json.dump({
                    "adapter_name": adapter_name,
                    "epochs": epochs,
                    "lr": lr,
                    "system_prompt": system_prompt,
                    "best_metric": float(best_metric),
                    "stopped_epoch": epoch + 1,
                    "cfg": asdict(CFG),
                }, f, ensure_ascii=False, indent=2)
            print(f"  saved best adapter to {save_dir}")
            # best model을 Google Drive에 백업
            if CFG.gdrive_backup_best:
                _backup_to_gdrive(save_dir, "best")
        else:
            patience_counter += 1
            print(f"  no improvement (patience {patience_counter}/{CFG.early_stopping_patience})")
            if valid_loader is not None and CFG.early_stopping_patience > 0 and patience_counter >= CFG.early_stopping_patience:
                print(f"  ** early stopping at epoch {epoch+1} (best_metric={best_metric:.5f}) **")
                stopped_early = True
                break

    if not stopped_early:
        print(f"[{adapter_name}] completed all {epochs} epochs (best_metric={best_metric:.5f})")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return str(save_dir)

In [ ]:
# ============ adapter 학습 실행 (세션 복구용) ============
# shared_adapter, count_adapter는 이전 세션에서 학습 완료 + Google Drive 백업됨
# material_adapter만 새로 학습

import shutil

def restore_adapter_from_gdrive(adapter_name: str) -> str:
    """Google Drive에서 adapter 복구. 없으면 None 반환."""
    local_dir = str(Path(CFG.output_dir) / adapter_name)
    gdrive_path = Path(CFG.gdrive_backup_dir) / adapter_name / "best"
    if gdrive_path.exists():
        Path(local_dir).mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(gdrive_path), local_dir, dirs_exist_ok=True)
        print(f"{adapter_name} restored from Google Drive: {gdrive_path}")
        print(f"  -> {local_dir}")
        return local_dir
    print(f"WARNING: {adapter_name} backup not found at {gdrive_path}")
    return None

# ---- shared_adapter: Drive 복구 또는 재학습 ----
shared_adapter_dir = restore_adapter_from_gdrive("shared_adapter")
if shared_adapter_dir is None:
    print("shared_adapter를 처음부터 학습합니다...")
    shared_adapter_dir = train_one_adapter(
        train_pool_df=shared_train_df,
        adapter_name="shared_adapter",
        system_prompt=SYSTEM_PROMPT_GENERAL,
        epochs=CFG.shared_epochs,
        lr=CFG.lr_shared,
        valid_df=valid_df,
    )

# ---- count_adapter: Drive 복구 또는 재학습 ----
count_adapter_dir = restore_adapter_from_gdrive("count_adapter")
if count_adapter_dir is None:
    print("count_adapter를 처음부터 학습합니다...")
    count_valid_df = None if valid_df is None else valid_df[valid_df["qtype"] == "count"].reset_index(drop=True)
    count_adapter_dir = train_one_adapter(
        train_pool_df=count_train_df,
        adapter_name="count_adapter",
        system_prompt=SYSTEM_PROMPT_COUNT,
        epochs=CFG.count_epochs,
        lr=CFG.lr_count,
        valid_df=count_valid_df if (count_valid_df is not None and len(count_valid_df)) else None,
    )

# ---- material_adapter: Drive 복구 또는 재학습 ----
material_adapter_dir = restore_adapter_from_gdrive("material_adapter")
if material_adapter_dir is None:
    print("material_adapter를 처음부터 학습합니다...")
    material_valid_df = None if valid_df is None else valid_df[valid_df["qtype"].isin(["material", "object"])].reset_index(drop=True)
    material_adapter_dir = train_one_adapter(
        train_pool_df=material_train_df,
        adapter_name="material_adapter",
        system_prompt=SYSTEM_PROMPT_MATERIAL,
        epochs=CFG.material_epochs,
        lr=CFG.lr_material,
        valid_df=material_valid_df if (material_valid_df is not None and len(material_valid_df)) else None,
    )

print("\nshared_adapter_dir  :", shared_adapter_dir)
print("count_adapter_dir   :", count_adapter_dir)
print("material_adapter_dir:", material_adapter_dir)

In [ ]:
# ============ OCR + 추론/앙상블 ============
from rapidfuzz import fuzz

ocr_reader = None
if CFG.use_ocr:
    try:
        import easyocr
        ocr_reader = easyocr.Reader(list(CFG.ocr_langs), gpu=torch.cuda.is_available())
    except Exception as e:
        print("easyocr init skipped:", repr(e))

def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

def normalize_for_ocr(x: str) -> str:
    x = str(x).lower()
    x = re.sub(r"[^0-9a-z가-힣]+", "", x)
    return x

def ocr_option_probs(image: Image.Image, row: pd.Series) -> np.ndarray:
    if ocr_reader is None:
        return np.ones(4, dtype=np.float32) / 4.0

    try:
        texts = ocr_reader.readtext(np.array(image), detail=0, paragraph=False)
    except Exception:
        return np.ones(4, dtype=np.float32) / 4.0

    merged = " ".join([str(t) for t in texts])
    norm_merged = normalize_for_ocr(merged)

    scores = np.zeros(4, dtype=np.float32)
    for idx, opt in enumerate([row["a"], row["b"], row["c"], row["d"]]):
        opt_norm = normalize_for_ocr(opt)
        if not opt_norm:
            continue

        if opt_norm in norm_merged:
            scores[idx] += 3.0

        nums = re.findall(r"\d+(?:\.\d+)?", opt_norm)
        if nums and all(n in norm_merged for n in nums):
            scores[idx] += 1.5

        # 부분 일치
        if norm_merged:
            scores[idx] += fuzz.partial_ratio(opt_norm, norm_merged) / 100.0

    if float(scores.max()) <= 0:
        return np.ones(4, dtype=np.float32) / 4.0
    return softmax_np(scores)

def build_prefix_messages_for_row(row: pd.Series, options: List[str], system_prompt: str, image: Image.Image):
    user_text = build_user_prompt(row["question"], options)
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text},
        ]},
    ]

def score_letters_with_logprob(
    model: PeftModel,
    processor: Any,
    image: Image.Image,
    prefix_messages: List[Dict[str, Any]],
    candidates: List[str] = ["a", "b", "c", "d"],
) -> np.ndarray:
    prefix_text = safe_apply_chat_template(processor, prefix_messages, tokenize=False, add_generation_prompt=True)
    full_texts = [prefix_text + c for c in candidates]

    prefix_inputs = processor(text=[prefix_text], images=[image], return_tensors="pt")
    prefix_inputs = move_batch_to_device(prefix_inputs, device)
    prefix_len = int(prefix_inputs["attention_mask"].sum().item())

    batch_inputs = processor(
        text=full_texts,
        images=[image] * len(candidates),
        padding=True,
        return_tensors="pt",
    )
    batch_inputs = move_batch_to_device(batch_inputs, device)

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            outputs = model(**batch_inputs)
            logits = outputs.logits

    scores = []
    attn = batch_inputs["attention_mask"]
    input_ids = batch_inputs["input_ids"]

    for i in range(len(candidates)):
        full_len = int(attn[i].sum().item())
        target_ids = input_ids[i, prefix_len:full_len]
        pred_logits = logits[i, prefix_len - 1: full_len - 1, :]
        log_probs = F.log_softmax(pred_logits, dim=-1)
        gathered = log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
        scores.append(float(gathered.sum().item()))

    return softmax_np(np.array(scores, dtype=np.float32))

def score_fulltext_with_logprob(
    model: PeftModel,
    processor: Any,
    image: Image.Image,
    prefix_messages: List[Dict[str, Any]],
    choice_texts: List[str],  # e.g., ["유리", "금속", "종이", "플라스틱"]
    normalize: bool = True,    # True = mean-logprob (length-normalized), False = sum
) -> np.ndarray:
    """Full choice text log-probability scoring with length normalization."""
    prefix_text = safe_apply_chat_template(processor, prefix_messages, tokenize=False, add_generation_prompt=True)
    full_texts = [prefix_text + t for t in choice_texts]

    prefix_inputs = processor(text=[prefix_text], images=[image], return_tensors="pt")
    prefix_inputs = move_batch_to_device(prefix_inputs, device)
    prefix_len = int(prefix_inputs["attention_mask"].sum().item())

    batch_inputs = processor(
        text=full_texts,
        images=[image] * len(choice_texts),
        padding=True,
        return_tensors="pt",
    )
    batch_inputs = move_batch_to_device(batch_inputs, device)

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            outputs = model(**batch_inputs)
            logits = outputs.logits

    scores = []
    attn = batch_inputs["attention_mask"]
    input_ids = batch_inputs["input_ids"]

    for i in range(len(choice_texts)):
        full_len = int(attn[i].sum().item())
        target_ids = input_ids[i, prefix_len:full_len]
        n_tokens = len(target_ids)
        if n_tokens == 0:
            scores.append(-1e9)
            continue
        pred_logits = logits[i, prefix_len - 1: full_len - 1, :]
        log_probs = F.log_softmax(pred_logits, dim=-1)
        gathered = log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
        total = float(gathered.sum().item())
        scores.append(total / n_tokens if normalize else total)

    return softmax_np(np.array(scores, dtype=np.float32))

def score_row_with_permutations(
    model: PeftModel,
    processor: Any,
    row: pd.Series,
    adapter_name: str,
    system_prompt: str,
    n_perm: int,
    seed: int = 42,
) -> np.ndarray:
    model.set_adapter(adapter_name)
    image = Image.open(row["path"]).convert("RGB")
    rng = random.Random(seed + int(hashlib.md5(str(row["id"]).encode()).hexdigest(), 16) % 10_000)  # [Fix 3] 결정적 해시

    options_original = [row["a"], row["b"], row["c"], row["d"]]
    accum = np.zeros(4, dtype=np.float32)

    for perm_idx in range(n_perm):
        perm = [0, 1, 2, 3]
        if perm_idx > 0:
            rng.shuffle(perm)

        options_perm = [options_original[i] for i in perm]
        prefix_messages = build_prefix_messages_for_row(row, options_perm, system_prompt, image)
        if CFG.use_fulltext_logprob:
            choice_texts = [options_perm[i] for i in range(4)]
            probs_perm = score_fulltext_with_logprob(model, processor, image, prefix_messages, choice_texts=choice_texts)
        else:
            probs_perm = score_letters_with_logprob(model, processor, image, prefix_messages, candidates=["a","b","c","d"])

        probs_orig = np.zeros(4, dtype=np.float32)
        for new_idx, p in enumerate(probs_perm):
            old_idx = perm[new_idx]
            probs_orig[old_idx] += float(p)

        accum += probs_orig

    accum = accum / max(1, n_perm)
    return accum / accum.sum()

def margin_of_probs(probs: np.ndarray) -> float:
    order = np.sort(probs)[::-1]
    return float(order[0] - order[1])

def combine_probs(row: pd.Series, shared_probs: np.ndarray, count_probs: Optional[np.ndarray],
                  text_probs: np.ndarray, ocr_probs: Optional[np.ndarray],
                  material_probs: Optional[np.ndarray] = None) -> np.ndarray:
    qtype = row["qtype"]
    w = CFG.ensemble_weights.get(qtype, CFG.ensemble_weights["default"])

    final = np.zeros(4, dtype=np.float32)
    final += w.get("shared", 0.0) * shared_probs
    if count_probs is not None:
        final += w.get("count", 0.0) * count_probs
    final += w.get("text", 0.0) * text_probs
    if ocr_probs is not None:
        final += w.get("ocr", 0.0) * ocr_probs
    if material_probs is not None:
        final += w.get("material", 0.0) * material_probs

    if final.sum() <= 0:
        final = shared_probs.copy()
    final = final / final.sum()
    return final

# ---- 모델 로드 (base 1개 + adapter 2개) ----
processor_inf = build_processor(train=False, ocr_mode=False)
processor_inf_ocr = build_processor(train=False, ocr_mode=True)

base_model = load_base_model(for_train=False, use_4bit=CFG.infer_use_4bit, dtype_name=CFG.infer_dtype)
infer_model = PeftModel.from_pretrained(base_model, shared_adapter_dir, adapter_name="shared")
infer_model.load_adapter(count_adapter_dir, adapter_name="count")
infer_model.load_adapter(material_adapter_dir, adapter_name="material")
infer_model.eval()
print("loaded adapters:", infer_model.peft_config.keys())

def optimize_ensemble_weights(val_pred_df: pd.DataFrame, qtypes: Optional[List[str]] = None):
    """Per-qtype ensemble weight optimization using scipy.optimize.
    val_pred_df must contain columns: qtype, answer, shared_probs, count_probs, text_probs, ocr_probs, material_probs
    """
    from scipy.optimize import minimize

    if qtypes is None:
        qtypes = val_pred_df["qtype"].unique().tolist()

    optimized = {}
    for qt in qtypes:
        mask = val_pred_df["qtype"] == qt
        sub = val_pred_df[mask]
        if len(sub) < 5:
            optimized[qt] = CFG.ensemble_weights.get(qt, CFG.ensemble_weights["default"])
            continue

        channels = ["shared", "count", "text", "ocr", "material"]
        probs_stack = {}
        for ch in channels:
            col = f"{ch}_probs"
            if col in sub.columns and sub[col].apply(lambda x: x is not None).any():
                probs_stack[ch] = np.stack(sub[col].values)
            else:
                probs_stack[ch] = np.zeros((len(sub), 4))

        true_labels = np.array([LETTERS.index(a) for a in sub["answer"]])

        def neg_accuracy(w):
            w_dict = {ch: max(0, w[i]) for i, ch in enumerate(channels)}
            total = sum(w_dict.values())
            if total <= 0:
                return 1.0
            blended = sum(w_dict[ch] * probs_stack[ch] for ch in channels) / total
            preds = blended.argmax(axis=1)
            return -np.mean(preds == true_labels)

        w0 = [CFG.ensemble_weights.get(qt, CFG.ensemble_weights["default"]).get(ch, 0.0) for ch in channels]
        result = minimize(neg_accuracy, w0, method="Nelder-Mead",
                          options={"maxiter": 500, "xatol": 0.01, "fatol": 0.001})

        opt_w = {ch: max(0, round(result.x[i], 3)) for i, ch in enumerate(channels)}
        total = sum(opt_w.values())
        if total > 0:
            opt_w = {ch: round(v / total, 3) for ch, v in opt_w.items()}
        optimized[qt] = opt_w
        acc = -result.fun
        old_acc = -neg_accuracy(w0)
        print(f"  {qt:15s}: {old_acc:.3f} -> {acc:.3f} ({acc - old_acc:+.3f})  w={opt_w}")

    return optimized

def predict_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    text_probs_all = predict_text_proba(df)

    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="predict")):
        row = row.copy()

        # shared
        shared_processor = processor_inf_ocr if row["qtype"] == "ocr" else processor_inf
        shared_probs = score_row_with_permutations(
            infer_model,
            shared_processor,
            row,
            adapter_name="shared",
            system_prompt=SYSTEM_PROMPT_GENERAL,
            n_perm=CFG.n_perm_shared,
            seed=CFG.seed,
        )

        # count expert
        count_probs = None
        if row["qtype"] == "count":
            count_probs = score_row_with_permutations(
                infer_model,
                processor_inf,
                row,
                adapter_name="count",
                system_prompt=SYSTEM_PROMPT_COUNT,
                n_perm=CFG.n_perm_count,
                seed=CFG.seed + 999,
            )
            # 불확실 count는 고해상도 1회 추가
            if margin_of_probs(count_probs) < CFG.uncertain_margin:
                hi_probs = score_row_with_permutations(
                    infer_model,
                    processor_inf_ocr,
                    row,
                    adapter_name="count",
                    system_prompt=SYSTEM_PROMPT_COUNT,
                    n_perm=1,
                    seed=CFG.seed + 777,
                )
                count_probs = 0.6 * count_probs + 0.4 * hi_probs
                count_probs = count_probs / count_probs.sum()

        # material/object expert
        material_probs = None
        if row["qtype"] in ("material", "object", "color"):
            material_probs = score_row_with_permutations(
                infer_model,
                processor_inf,
                row,
                adapter_name="material",
                system_prompt=SYSTEM_PROMPT_MATERIAL,
                n_perm=CFG.n_perm_material,
                seed=CFG.seed + 555,
            )

        # text prior
        text_probs = text_probs_all[i]

        # OCR heuristic
        ocr_probs = None
        if row["qtype"] == "ocr":
            img = Image.open(row["path"]).convert("RGB")
            ocr_probs = ocr_option_probs(img, row)

        final_probs = combine_probs(row, shared_probs, count_probs, text_probs, ocr_probs, material_probs)
        pred = LETTERS[int(np.argmax(final_probs))]

        results.append({
            "id": row["id"],
            "answer": pred,
            "qtype": row["qtype"],
            "p_a": float(final_probs[0]),
            "p_b": float(final_probs[1]),
            "p_c": float(final_probs[2]),
            "p_d": float(final_probs[3]),
            "shared_margin": margin_of_probs(shared_probs),
            # [Fix 5] branch별 probs 저장 — optimize_ensemble_weights 연결용
            "shared_probs": shared_probs.tolist(),
            "count_probs": count_probs.tolist() if count_probs is not None else None,
            "text_probs": text_probs.tolist(),
            "ocr_probs": ocr_probs.tolist() if ocr_probs is not None else None,
            "material_probs": material_probs.tolist() if material_probs is not None else None,
        })

    return pd.DataFrame(results)

# optional: holdout 평가
if valid_df is not None and len(valid_df):
    valid_pred_df = predict_dataframe(valid_df)
    valid_eval = valid_df[["id", "answer"]].merge(valid_pred_df[["id", "answer"]], on="id", suffixes=("_true", "_pred"))
    valid_acc = (valid_eval["answer_true"] == valid_eval["answer_pred"]).mean()
    print("holdout accuracy (before optimize):", round(valid_acc, 5))

    # [Fix 5] ensemble weight optimization 연결
    print("\n--- ensemble weight optimization ---")
    # list → numpy array 변환
    _opt_df = valid_df[["id", "answer", "qtype"]].merge(valid_pred_df.drop(columns=["answer"]), on="id")
    for ch in ["shared", "count", "text", "ocr", "material"]:
        col = f"{ch}_probs"
        if col in _opt_df.columns:
            _opt_df[col] = _opt_df[col].apply(lambda x: np.array(x) if isinstance(x, list) else (np.zeros(4) if x is None else x))
    try:
        optimized_weights = optimize_ensemble_weights(_opt_df)
        for qt, w in optimized_weights.items():
            CFG.ensemble_weights[qt] = w
        print("applied optimized weights to CFG.ensemble_weights")

        # 최적 가중치로 holdout 재평가
        _reeval = []
        for _, r in _opt_df.iterrows():
            sp = np.array(r["shared_probs"]); cp = np.array(r["count_probs"])
            tp = np.array(r["text_probs"]); op = np.array(r["ocr_probs"])
            mp = np.array(r["material_probs"])
            fp = combine_probs(r, sp, cp if r["qtype"] == "count" else None, tp,
                               op if r["qtype"] == "ocr" else None,
                               mp if r["qtype"] in ("material", "object", "color") else None)
            _reeval.append(LETTERS[int(np.argmax(fp))])
        reeval_acc = np.mean([p == t for p, t in zip(_reeval, _opt_df["answer"])])
        print(f"holdout accuracy (after optimize): {reeval_acc:.5f} (delta: {reeval_acc - valid_acc:+.5f})")
    except Exception as e:
        print(f"optimize failed (using default weights): {repr(e)}")

    display(valid_pred_df.head(3))
    valid_pred_df.to_csv(Path(CFG.output_dir) / "valid_predictions.csv", index=False)

In [ ]:
# ============ test 추론 + submission 저장 ============
test_pred_df = predict_dataframe(test_df)
submission = test_pred_df[["id", "answer"]].copy()

submission_path = Path(CFG.output_dir) / "submission.csv"
proba_path = Path(CFG.output_dir) / "test_predictions_with_probs.csv"

submission.to_csv(submission_path, index=False)
test_pred_df.to_csv(proba_path, index=False)

print("saved:", submission_path)
print("saved:", proba_path)
display(submission.head())

# Google Drive 자동 백업
backup_dir = Path(CFG.gdrive_backup_dir) / "submissions"
backup_dir.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy2(submission_path, backup_dir / "submission.csv")
shutil.copy2(proba_path, backup_dir / "test_predictions_with_probs.csv")
print(f"backed up to {backup_dir}")